# Part 1 - Data Collection

In [1]:
# some preparations
import yfinance as yf
import pandas as pd
import numpy as np
from io import StringIO
import requests
import os

第一步：从维基百科的表格中提取出股票代码列的数据

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

response = requests.get(
    "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
    headers=headers
)

data = pd.read_html(StringIO(response.text))

df = data[0]
df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [3]:
tickers = df["Symbol"].str.replace('.', '-', regex=False).tolist()
tickers[: 10]

['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']

第二步：用股票代码 `tickers` 列作为匹配源获取2015年1月-2025年6月的对应股票数据信息

In [4]:
# 下载不稳定，这里下载时我写了个函数用以重复下载之前下载失败的数据。
#  原语句：raw = yf.download(tickers, start="2015-01-01", end="2025-06-30", progress=True, auto_adjust=False)
def download_sp500(tickers, start="2015-01-01", end="2025-06-30"):
    # 第一次批量下载
    raw = yf.download(tickers, start=start, end=end, progress=True, auto_adjust=False)
    
    # 找出失败的（整列全是NaN）
    all_tickers = raw.columns.get_level_values("Ticker").unique().tolist()
    failed = [t for t in all_tickers if raw.xs(t, level="Ticker", axis=1).isna().all().all()]
    missing = [t for t in tickers if t not in all_tickers]
    retry = list(set(failed + missing))
    
    print(f"第一次下载: 成功 {len(tickers)-len(retry)} 只, 失败 {len(retry)} 只")
    
    # 对失败的逐只重试一次
    if retry:
        for ticker in retry:
            try:
                tmp = yf.download(ticker, start=start, end=end, progress=False)
                if not tmp.empty:
                    tmp.columns = pd.MultiIndex.from_product([[c for c in tmp.columns], [ticker]], names=["Price", "Ticker"])
                    raw = pd.concat([raw, tmp], axis=1)
                    print(f"{ticker} 重试成功")
            except:
                print(f"{ticker} 放弃")
    
    # 去重列
    raw = raw.loc[:, ~raw.columns.duplicated()]
    return raw

raw = download_sp500(tickers)
raw.to_parquet("../data/raw_data.parquet")
print(f"保存成功, shape: {raw.shape}")

[**********************64%******                 ]  324 of 503 completed$Q: possibly delisted; no price data found  (1d 2015-01-01 -> 2025-06-30) (Yahoo error = "Data doesn't exist for startDate = 1420088400, endDate = 1751256000")
[*********************100%***********************]  503 of 503 completed

1 Failed download:
['Q']: possibly delisted; no price data found  (1d 2015-01-01 -> 2025-06-30) (Yahoo error = "Data doesn't exist for startDate = 1420088400, endDate = 1751256000")


第一次下载: 成功 502 只, 失败 1 只


$Q: possibly delisted; no price data found  (1d 2015-01-01 -> 2025-06-30) (Yahoo error = "Data doesn't exist for startDate = 1420088400, endDate = 1751256000")

1 Failed download:
['Q']: possibly delisted; no price data found  (1d 2015-01-01 -> 2025-06-30) (Yahoo error = "Data doesn't exist for startDate = 1420088400, endDate = 1751256000")


保存成功, shape: (2637, 3018)


反复试了多次，这个代码是Q的怎么也提取不了。似乎不在yf库中。就不考虑它了。

第三步：数据清洗

In [5]:
# 去掉多重column
stocks = raw.stack(level='Ticker').reset_index()
print(stocks.shape)
stocks.head()

(1326411, 8)


Price,Date,Ticker,Adj Close,Close,High,Low,Open,Volume
0,2015-01-02,A,37.054733,40.560001,41.310001,40.369999,41.180000,1529200.0
1,2015-01-02,AAPL,24.214886,27.332500,27.860001,26.837500,27.847500,212818400.0
2,2015-01-02,ABBV,41.755470,65.889999,66.400002,65.440002,65.440002,5086100.0
3,2015-01-02,ABNB,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-01-02,ABT,36.235119,44.900002,45.450001,44.639999,45.250000,3216600.0


In [6]:
# 去掉index列的column name
stocks.columns.name = None
# 去掉有NaN值的相关行数据
stocks = stocks.dropna(subset=['Close'])
# 跟pdf示例保持一致，将Ticker小写
stocks['Ticker'] = stocks['Ticker'].str.lower()
stocks.head()

,Date,Ticker,Adj Close,Close,High,Low,Open,Volume
0,2015-01-02,a,37.054733,40.560001,41.310001,40.369999,41.180000,1529200.0
1,2015-01-02,aapl,24.214886,27.332500,27.860001,26.837500,27.847500,212818400.0
2,2015-01-02,abbv,41.755470,65.889999,66.400002,65.440002,65.440002,5086100.0
4,2015-01-02,abt,36.235119,44.900002,45.450001,44.639999,45.250000,3216600.0
5,2015-01-02,acgl,18.539352,19.496668,19.860001,19.426666,19.733334,1101600.0


In [7]:
stocks.to_csv("../data/raw_stocks.csv")